In [1]:
import os
import pandas as pd
import numpy as np
from IPython.display import display

script_dir = os.getcwd() 

FINAL_COMP_DIR = os.path.join(script_dir, 'final_comparison')
os.makedirs(FINAL_COMP_DIR, exist_ok=True)
print(f"📁 Folder output untuk CSV disiapkan di: {FINAL_COMP_DIR}\n")

subjects = [f"SUB{i}" for i in range(1, 13)]

# PERBAIKAN: Menambahkan 'WER' tepat setelah 'cer' pada daftar metrik
metrics = ['cer', 'WER', 'BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4', 'ROUGE-1-P', 'ROUGE-1-R', 'ROUGE-1-F', 'BertScore']

print("Mengekstrak dan menghitung nilai seluruh metrik dari eksperimen Log-Mel Spectrogram...\n")

experiments = [
    {
        'Decoder': 'LSTM',
        'Train Data': 'Single',
        'type': 'single',
        'template': "complete_metrics_NOISE_BASELINE_{subject}_eq_3_0_log-mel_test_predictions_6_1.csv"
    },
    {
        'Decoder': 'IndoGPT',
        'Train Data': 'Single',
        'type': 'single',
        'template': "complete_metrics_NOISE_BASELINE_{subject}_eq_3_0_logmel_test_predictions_10_1_IndoGPT.csv"
    },
    {
        'Decoder': 'LSTM',
        'Train Data': 'All',
        'type': 'all',
        'template': "complete_metrics_NOISE_BASELINE_all_eq_3_0_log-mel_test_predictions_6_1.csv"
    },
    {
        'Decoder': 'IndoGPT',
        'Train Data': 'All',
        'type': 'all',
        'template': "complete_metrics_NOISE_BASELINE_all_eq_3_0_logmel_test_predictions_10_1_IndoGPT.csv"
    }
]

tables_data = {m: [] for m in metrics}

summary_rows = []


for exp in experiments:
    summary_row = {
        'Decoder': exp['Decoder'],
        'Train Data': exp['Train Data']
    }
    
    exp_metric_rows = {m: {'Decoder': exp['Decoder'], 'Train Data': exp['Train Data']} for m in metrics}
    
    raw_values = {m: [] for m in metrics}
    
    if exp['type'] == 'single':
        for subject in subjects:
            filename = exp['template'].format(subject=subject)
            filepath = os.path.join(script_dir, filename)
            
            if os.path.exists(filepath):
                try:
                    df = pd.read_csv(filepath)
                    for m in metrics:
                        if m in df.columns:
                            exp_metric_rows[m][subject] = round(df[m].mean(), 4)
                            raw_values[m].extend(df[m].dropna().tolist())
                        else:
                            exp_metric_rows[m][subject] = np.nan
                except Exception as e:
                    print(f"  [ERROR] Gagal membaca {filename}: {e}")
                    for m in metrics: exp_metric_rows[m][subject] = np.nan
            else:
                for m in metrics: exp_metric_rows[m][subject] = np.nan
                
    elif exp['type'] == 'all':
        filename = exp['template']
        filepath = os.path.join(script_dir, filename)
        
        if os.path.exists(filepath):
            try:
                df_all = pd.read_csv(filepath)
                for subject in subjects:
                    sub_df = df_all[df_all['subject'] == subject]
                    if len(sub_df) > 0:
                        for m in metrics:
                            if m in sub_df.columns:
                                exp_metric_rows[m][subject] = round(sub_df[m].mean(), 4)
                                raw_values[m].extend(sub_df[m].dropna().tolist())
                            else:
                                exp_metric_rows[m][subject] = np.nan
                    else:
                        for m in metrics: exp_metric_rows[m][subject] = np.nan
            except Exception as e:
                print(f"  [ERROR] Gagal membaca {filename}: {e}")
                for subject in subjects:
                    for m in metrics: exp_metric_rows[m][subject] = np.nan
        else:
            print(f"  ⚠️ [INFO] File {filename} belum ditemukan.")
            for subject in subjects:
                for m in metrics: exp_metric_rows[m][subject] = np.nan

    for m in metrics:
        if raw_values[m]:
            global_avg = round(np.mean(raw_values[m]), 4)
        else:
            global_avg = np.nan
            
        exp_metric_rows[m]['Rata-rata Global'] = global_avg
        tables_data[m].append(exp_metric_rows[m])
        
        summary_row[m] = global_avg
        
    summary_rows.append(summary_row)

column_order = ['Decoder', 'Train Data'] + subjects + ['Rata-rata Global']

for m in metrics:
    df_metric = pd.DataFrame(tables_data[m])
    df_metric = df_metric[[col for col in column_order if col in df_metric.columns]]
    
    print("=" * 110)
    print(f"TABEL PERBANDINGAN PERFORMA: {m.upper()}")
    print("=" * 110)
    display(df_metric)
    
    metric_csv_path = os.path.join(FINAL_COMP_DIR, f"log-mel_noise_{m}.csv")
    df_metric.to_csv(metric_csv_path, index=False)
    print(f"✅ Tersimpan: {metric_csv_path}\n")

df_summary_global = pd.DataFrame(summary_rows)

summary_col_order = ['Decoder', 'Train Data'] + metrics
df_summary_global = df_summary_global[[col for col in summary_col_order if col in df_summary_global.columns]]

print("*" * 110)
print("🏆 TABEL GRAND SUMMARY: RATA-RATA GLOBAL SELURUH METRIK 🏆")
print("*" * 110)
display(df_summary_global)

summary_csv_path = os.path.join(FINAL_COMP_DIR, "log-mel_noise_grand_summary.csv")
df_summary_global.to_csv(summary_csv_path, index=False)
print(f"✅ Tersimpan: {summary_csv_path}\n")


CURRENT_DIR = os.getcwd()
DATASET_DIR = os.path.abspath(os.path.join(CURRENT_DIR, '../../../dataset'))

oov_summary_list = []

print("\n\n" + "=" * 110)
print("MEMULAI ANALISIS OOV KARAKTER...")
print("=" * 110)

for subject in subjects:
    train_file = os.path.join(DATASET_DIR, f"{subject}_eq_3_0_train.csv")
    val_file = os.path.join(DATASET_DIR, f"{subject}_eq_3_0_val.csv")
    test_file = os.path.join(DATASET_DIR, f"{subject}_eq_3_0_test.csv")
    
    if os.path.exists(train_file) and os.path.exists(val_file) and os.path.exists(test_file):
        df_train = pd.read_csv(train_file)
        df_val = pd.read_csv(val_file)
        df_test = pd.read_csv(test_file)
        
        # Gunakan astype(str) untuk mencegah error jika ada data null/kosong
        train_text = "".join(df_train['sentence'].astype(str).tolist())
        test_text = "".join(df_test['sentence'].astype(str).tolist())
        
        train_chars = set(train_text)
        test_chars = set(test_text)
        
        oov_chars = test_chars - train_chars
        
        test_text_len = len(test_text)
        if test_text_len > 0:
            oov_occurrences = sum(1 for char in test_text if char in oov_chars)
            oov_rate = (oov_occurrences / test_text_len) * 100
        else:
            oov_occurrences = 0
            oov_rate = 0.0

        oov_summary_list.append({
            'Subject': subject,
            'Train (Baris)': len(df_train),
            'Val (Baris)': len(df_val),
            'Test (Baris)': len(df_test),
            'Kamus Train': len(train_chars),
            'Kamus Test': len(test_chars),
            'Karakter OOV': "".join(sorted(list(oov_chars))) if oov_chars else "-",
            'Total Muncul OOV': oov_occurrences,
            'OOV Rate (%)': round(oov_rate, 4)
        })
    else:
        # Jika salah satu dari 3 file per subjek tidak ada, berikan peringatan
        print(f"  ⚠️ Melewati {subject}: File train/val/test tidak lengkap di folder dataset.")

if oov_summary_list:
    df_summary = pd.DataFrame(oov_summary_list)

    print("\n" + "=" * 110)
    print("RINGKASAN DATA SPLIT & OOV KARAKTER PER SUBJEK")
    print("=" * 110)
    display(df_summary)
    
    oov_csv_path = os.path.join(FINAL_COMP_DIR, "oov_tokenizer_char.csv")
    df_summary.to_csv(oov_csv_path, index=False)
    print(f"✅ Tersimpan: {oov_csv_path}\n")
else:
    print("❌ Tidak ada data yang bisa ditampilkan. Pastikan file *_eq_3_0_*.csv sudah dibuat di dalam folder /dataset.")

📁 Folder output untuk CSV disiapkan di: d:\GitRepos\EEG-to-Text_Conformer-Transducer\src\pipelines\training\final_comparison

Mengekstrak dan menghitung nilai seluruh metrik dari eksperimen Log-Mel Spectrogram...

TABEL PERBANDINGAN PERFORMA: CER


,Decoder,Train Data,SUB1,SUB2,SUB3,SUB4,SUB5,SUB6,SUB7,SUB8,SUB9,SUB10,SUB11,SUB12,Rata-rata Global
0,LSTM,Single,77.6793,76.9440,77.6176,75.4545,75.9556,77.3263,78.9733,74.5139,80.5646,75.2712,78.1985,85.1604,77.3281
1,IndoGPT,Single,87.7171,78.4477,74.7456,80.2511,74.7797,79.5021,72.8062,72.4702,75.4334,75.7538,77.2513,82.6389,77.5188
2,LSTM,All,76.4449,73.5754,72.2352,73.1578,74.0310,76.2211,77.7345,72.6032,74.5913,75.6020,73.0007,71.3335,74.6120
3,IndoGPT,All,74.1153,70.2708,80.8467,75.5702,72.6871,74.5613,74.2522,71.3450,79.0202,70.4363,73.7568,78.8690,74.3066


✅ Tersimpan: d:\GitRepos\EEG-to-Text_Conformer-Transducer\src\pipelines\training\final_comparison\log-mel_noise_cer.csv

TABEL PERBANDINGAN PERFORMA: WER


,Decoder,Train Data,SUB1,SUB2,SUB3,SUB4,SUB5,SUB6,SUB7,SUB8,SUB9,SUB10,SUB11,SUB12,Rata-rata Global
0,LSTM,Single,100.1190,98.0000,100.0000,95.9167,98.0000,102.5000,102.9167,95.9524,100.0000,101.7544,99.1597,108.3333,99.7909
1,IndoGPT,Single,132.2500,94.5714,100.0000,118.9167,101.3333,102.9167,98.4940,97.3214,97.8947,101.7544,97.9832,103.3333,104.9175
2,LSTM,All,100.9048,101.0714,95.8333,98.5833,103.3333,99.2857,103.4524,98.7500,101.0526,100.2632,100.7843,100.0000,100.3187
3,IndoGPT,All,96.9881,95.0000,103.0000,101.0833,96.5556,100.6667,101.2857,94.5357,105.0877,94.0351,98.1793,108.3333,99.0606


✅ Tersimpan: d:\GitRepos\EEG-to-Text_Conformer-Transducer\src\pipelines\training\final_comparison\log-mel_noise_WER.csv

TABEL PERBANDINGAN PERFORMA: BLEU-1


...

✅ Tersimpan: d:\GitRepos\EEG-to-Text_Conformer-Transducer\src\pipelines\training\final_comparison\log-mel_noise_BertScore.csv

**************************************************************************************************************
🏆 TABEL GRAND SUMMARY: RATA-RATA GLOBAL SELURUH METRIK 🏆
**************************************************************************************************************


,Decoder,Train Data,cer,WER,BLEU-1,BLEU-2,BLEU-3,BLEU-4,ROUGE-1-P,ROUGE-1-R,ROUGE-1-F,BertScore
0,LSTM,Single,77.3281,99.7909,1.5897,0.5821,0.4872,0.4523,3.7566,2.0433,2.5683,44.3668
1,IndoGPT,Single,77.5188,104.9175,3.4975,1.4196,1.1068,1.0107,6.8090,4.5213,5.1243,47.4769
2,LSTM,All,74.6120,100.3187,3.6286,1.6585,1.4263,1.2540,5.6526,4.0955,4.6469,47.6658
3,IndoGPT,All,74.3066,99.0606,6.1297,2.8017,2.2533,2.0908,8.5979,6.5359,7.3581,53.5146


✅ Tersimpan: d:\GitRepos\EEG-to-Text_Conformer-Transducer\src\pipelines\training\final_comparison\log-mel_noise_grand_summary.csv



MEMULAI ANALISIS OOV KARAKTER...

RINGKASAN DATA SPLIT & OOV KARAKTER PER SUBJEK


,Subject,Train (Baris),Val (Baris),Test (Baris),Kamus Train,Kamus Test,Karakter OOV,Total Muncul OOV,OOV Rate (%)
0,SUB1,70,10,20,24,23,-,0,0.0000
1,SUB2,35,5,10,23,23,z,1,0.3125
2,SUB3,35,5,10,23,21,-,0,0.0000
3,SUB4,70,10,20,24,22,-,0,0.0000
4,SUB5,35,5,10,23,23,-,0,0.0000
5,SUB6,70,10,20,23,22,-,0,0.0000
6,SUB7,70,10,20,24,22,-,0,0.0000
7,SUB8,70,10,20,23,23,-,0,0.0000
8,SUB9,64,9,19,24,22,-,0,0.0000
9,SUB10,63,9,19,23,23,-,0,0.0000


✅ Tersimpan: d:\GitRepos\EEG-to-Text_Conformer-Transducer\src\pipelines\training\final_comparison\oov_tokenizer_char.csv

